# 14.a — Ad hoc test: how far is today's cropland from the suitability threshold, in °C and in % of rain?

**Ad hoc test, outside the suitability chain**: it reads the ERA5 BIO variables of 1.1.d, the SVM and its fold models
(3.g, 4.g), the delta BIO variables of the runs (1.3.a) and the final suitability of 6.b, and writes only in
`<out_root>/test_climate_margin/<route>/<set>/`. Nothing of the chain is changed.

**Question.** 6.b says whether a cell is suitable in each climate; 12.e and 12.f measure how close it is to the
threshold in units of the SVM probability. A cell at p = 0.9 can be 0.5 °C from the threshold on a steep part of the
response and a cell at p = 0.6 can be 3 °C from it on a gentle part. How much warming, cooling, drying or wetting does
each cell of today's cropland tolerate before it becomes unsuitable, how sharp is the threshold, and is the change of
each run (the warming, the AMOC weakening) beyond this margin?

**Method.**
- **Domain**: the cells where the crop is grown today (MIRCA2000 presences of the training table) and that are
  suitable on ERA5 (final rule of 6.b, high input). The cells the model misses on ERA5 are left out, as in 6.b.
- **Uniform shifts of the ERA5 climate** of each cell (a sensitivity experiment, like the delta method with an
  artificial change): every temperature variable + ΔT (BIO01, 02, 06, 07, 11, 12; the growing-period temperature of the
  ECOCROP TMIN test too); the precipitation × (1 + ΔP) (BIO05, BIO10); the Hargreaves ET0 of each phase follows the
  temperature, ET0 × (1 + ΔT / (T̄ + 17.8)) with T̄ the mean temperature of the phase (exact for the daily formula when
  the radiation and the daily range do not change); radiation, calendar and soil unchanged.
  ΔT from −8 to +8 °C by 0.25 °C, ΔP from −60 to +60 % by 5 %, and a coarse two-dimensional grid (ΔT by 1 °C from −6 to
  +6, ΔP by 10 % from −40 to +40 %).
- **Suitability at each shift**: the rule of 4.g / 6.b (probability at equal priors of the fold model that did not see
  the cell, ≥ 0.5; not novel: MESS, NT2, kernel support; growing-period temperature of the lower part of the cell ≥
  ECOCROP TMIN). The soil does not change with the climate. Check: at ΔT = ΔP = 0 the domain is suitable.
- **Margins** (per cell, outward from 0): the shift at which the cell first becomes unsuitable, on each side (cold /
  warm, dry / wet), and its cause: the probability (crossing of 0.5, interpolated between the grid points), the TMIN
  floor (exact) or a novel climate (the model has seen no such climate: the margin is then a lower bound of the real
  one). Beyond the scanned range: censored, set to the edge + one step in the tables.
- **Sharpness**: the slope of the probability where it crosses 0.5, expressed as the width in °C (or %) of a linear
  fall from 0.7 to 0.3; a width below 1 °C is a sharp threshold, the behaviour that can look like a tipping in crop
  viability.
- **Response curves**: probability (median and quartiles over the cells) and share of today's crop area still
  suitable as functions of ΔT and ΔP, per crop and region; the 2-D grid gives the share in the plane (ΔT, ΔP).
- **The runs against the margins**: the change of each run in the same units, from the delta BIO variables of 1.3.a
  (growing-period temperature, ΔT_run, and season precipitation, ΔP_run, against ERA5); a cell has *crossed* when
  ΔT_run or ΔP_run is beyond its margin, is *approaching* when it is within 1 °C or 10 % of it, *comfortable*
  otherwise; for the AMOC weakening, the cells that cross only with the hosing or only without it. Consistency: the
  share of the cells lost in 6.b that the margins predict (a run changes the variables unevenly, so the uniform
  margin explains part of the losses only).

**Reading.** A narrow transition (sharp threshold) with a small margin means that a modest change of the climate
switches the cell; a broad transition means a gradual degradation. The margins of a crop and region against ΔT_run of
the 2050 runs give the threshold in °C that the warming or the AMOC weakening crosses. These are thresholds of the
climate niche estimated by the SVM, not physiological limits measured on the crop; a uniform shift changes neither
the variability nor the shape of the season.

Outputs in `<out_root>/test_climate_margin/<route>/<set>/`: `margins_<crop>.nc` (per cell), `margin_summary.csv`,
`runs_vs_margin.csv`, `amoc_vs_margin.csv`, `curves_dT.csv`, `curves_dP.csv`, `grid2d.csv`, the figures.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore'); os.environ["PYTHONWARNINGS"] = "ignore"   # also in the joblib workers

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 1.3.a, 4.g and 6.b
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")
CELL_AREA_THR = 0.0005
CROPS = os.environ.get("MARGIN_CROPS", "Maize_rf,Potato_rf,Soybean_rf,Rice_rf,Wheat_rf,Wheat-winter_rf,Wheat-spring_rf").split(",")
ERA5_NAME, *ERA5_PERIOD = os.environ.get("MARGIN_ERA5", "ERA5,1981,2014").split(",")   # (local test: stand-in climate)
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVEL = "high"                    # input level of the soil in the domain (6.b)

# ---- shifts
DT_GRID = np.round(np.arange(-8.0, 8.0001, 0.25), 2)       # degC
DP_GRID = np.round(np.arange(-0.60, 0.60001, 0.05), 2)     # fraction of the precipitation
GRID2_DT = np.arange(-6.0, 6.0001, 1.0)
GRID2_DP = np.round(np.arange(-0.4, 0.40001, 0.1), 2)
DO_2D = os.environ.get("MARGIN_2D", "1") == "1"
NEAR_DT, NEAR_DP = 1.0, 0.10      # "approaching": within 1 degC or 10 % of the margin
SHARP_DT, SHARP_DP = 1.0, 0.10    # sharp threshold: fall of p from 0.7 to 0.3 within 1 degC / 10 %
KERNEL_EPS = 0.01
N_JOBS = min(5, os.cpu_count() or 1)
TEMP_COLS = ["BIO01", "BIO02", "BIO06", "BIO07", "BIO11", "BIO12"]
PR_COLS = ["BIO05", "BIO10"]
ET0_OF = {"BIO04": ("BIO01", "BIO02"), "BIO09": ("BIO06", "BIO07")}   # Hargreaves ET0 of each phase, its temperatures
CAUSES = {0: "beyond the range", 1: "probability", 2: "ECOCROP TMIN", 3: "novel climate"}

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
DELTA = Path(os.environ.get("MARGIN_DELTA") or cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET))
B6 = Path(os.environ.get("MARGIN_6B") or cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET))
OUT = Path(os.environ.get("MARGIN_OUT") or cu.out_dir(CFG, "test_climate_margin", ROUTE, SET))   # the only folder written
OUT.mkdir(parents=True, exist_ok=True)
SCEN = ["ERA5", *RUNS]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
DT_CELL = cu.tmin_warming(CFG)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
era5_file = lambda crop: ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
print(f"set {SET} | crops {CROPS} | dT {DT_GRID[0]}..{DT_GRID[-1]} by {DT_GRID[1] - DT_GRID[0]:g} | "
      f"dP {DP_GRID[0]:+.0%}..{DP_GRID[-1]:+.0%} | 2-D {DO_2D}\nwritten only in {OUT}")

## 1. The climate of the domain, shifted, and its suitability

In [ ]:
def season_mean(a, b, cal):
    # mean over the season of a phase-1 and a phase-2 value, weighted by the phase lengths (as cu.growing_temperature)
    l1 = np.mod(cal["ph2"] - cal["sow"], 365)
    l2 = np.mod(cal["mat"] - cal["ph2"], 365) + 1
    return np.where(l1 > 0, (l1 * a + l2 * b) / (l1 + l2), b)


def context(crop):
    # everything needed to predict the suitability of the domain cells of one crop under shifted climates
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, n1, n0 = bundle["bio_cols"], bundle["n_class1"], bundle["n_class0"]
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(era5_file(crop)) as ds:
        A = cu.bio_array(ds, cols, CFG)
        allv = {v: ds[v].values for v in ds.data_vars if v.startswith("BIO") and not v.endswith("e")}
        cal = {"sow": ds["sowing_doy"].values, "ph2": ds["phase2_start_doy"].values, "mat": ds["maturity_doy"].values}
        tg = np.asarray(cu.growing_temperature(ds, cu.ECOCROP_TMIN[crop][1]), float)
    REF = A[ii, jj]; REF = REF[np.isfinite(REF).all(axis=1)]
    F = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float),
                           tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
    with xr.open_dataset(B6 / f"climate_soil_scenarios_{crop}.nc") as d6:
        S6 = {s: np.nan_to_num(d6[f"suitable_{LEVEL}"].sel(scenario=s).values) == 1 for s in SCEN}
        N6 = {s: d6["novel"].sel(scenario=s).values == 1 for s in SCEN}
    pres = tr["label"].values == 1
    rows = np.nonzero(pres & S6["ERA5"][ii, jj] & np.isfinite(A[ii, jj]).all(axis=1))[0]   # domain: rows of the table
    I, J = ii[rows], jj[rows]
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)[I, J]
    return {"crop": crop, "cols": cols, "n1": n1, "n0": n0, "scaler": bundle["scaler"], "kfun": cu.kernel_support(bundle["model"]),
            "F": F, "REF": REF, "n_tr": len(tr), "rows": rows, "I": I, "J": J, "w": crop_ha,
            "V": {v: a[I, J] for v, a in allv.items()}, "cal": {k: v[I, J] for k, v in cal.items()},
            "tg": tg[I, J], "dt": DT_CELL[I, J], "tmin": cu.ECOCROP_TMIN[crop][0], "S6": S6, "N6": N6,
            "n_pres": int(pres.sum())}


def shifted(ctx, dT=0.0, dP=0.0):
    # BIO rows of the domain with every temperature + dT, the precipitation x (1 + dP), the Hargreaves ET0 following
    V, W = ctx["V"], dict(ctx["V"])
    for c in TEMP_COLS:
        if c in V:
            W[c] = V[c] + dT
    for c in PR_COLS:
        if c in V:
            W[c] = V[c] * (1 + dP)
    for e, (tx, tn) in ET0_OF.items():
        if e in V:
            tb = (V[tx] + V[tn]) / 2 + 17.8
            W[e] = np.where(tb > 1.0, V[e] * (1 + dT / np.where(tb > 1.0, tb, 1.0)), V[e])
    return np.column_stack([W[c] for c in ctx["cols"]])


def climate_at(ctx, dT=0.0, dP=0.0):
    # probability at equal priors (fold model that did not see the cell), novel, TMIN passed, suitable (rule of 4.g)
    X = shifted(ctx, dT, dP)
    full = np.full((ctx["n_tr"], X.shape[1]), np.nan)
    full[ctx["rows"]] = X
    p = cu.balanced_proba(cu.crossfit_proba(ctx["F"], full)[ctx["rows"]], ctx["n1"], ctx["n0"])
    nov = (cu.mess(X, ctx["REF"])[0] < 0) | (cu.nt2(X, ctx["REF"]) > 1) | (ctx["kfun"](ctx["scaler"].transform(X)) < KERNEL_EPS)
    tmin_ok = ctx["tg"] + dT + ctx["dt"] >= ctx["tmin"]
    return p, nov, tmin_ok, (p >= cu.CLIMATE_P) & ~nov & tmin_ok


def scan(ctx, grid, what):
    # the suitability along one axis (what = "dT" or "dP"): arrays (n_grid, n_cells)
    P, N, T, S = [], [], [], []
    for g in grid:
        p, nov, tok, s = climate_at(ctx, dT=g if what == "dT" else 0.0, dP=g if what == "dP" else 0.0)
        P.append(p); N.append(nov); T.append(tok); S.append(s)
    return {k: np.array(v) for k, v in zip("PNTS", (P, N, T, S))}


def margin(grid, R, side, ctx=None):
    # first shift outward from 0 on one side where the cell becomes unsuitable: value (NaN = beyond the range), cause,
    # width of a linear fall of p from 0.7 to 0.3 at the crossing (cause "probability" only)
    k0 = int(np.argmin(np.abs(grid)))
    idx = np.arange(k0, len(grid)) if side > 0 else np.arange(k0, -1, -1)
    g, S, P, N, T = grid[idx], R["S"][idx], R["P"][idx], R["N"][idx], R["T"][idx]
    n = S.shape[1]
    val, cause, width = np.full(n, np.nan), np.zeros(n, np.int8), np.full(n, np.nan)
    bad = ~S
    has = bad[1:].any(axis=0)
    k = np.where(has, np.argmax(bad[1:], axis=0) + 1, 0)
    for m in np.nonzero(has)[0]:
        kk = k[m]
        g0, g1, p0, p1 = g[kk - 1], g[kk], P[kk - 1, m], P[kk, m]
        if N[kk, m]:
            cause[m], val[m] = 3, (g0 + g1) / 2
        elif not T[kk, m]:
            cause[m] = 2
            val[m] = (ctx["tmin"] - ctx["tg"][m] - ctx["dt"][m]) if ctx is not None else (g0 + g1) / 2
        else:
            cause[m] = 1
            val[m] = g0 + (p0 - cu.CLIMATE_P) / max(p0 - p1, 1e-9) * (g1 - g0)
            slope = (p0 - p1) / abs(g1 - g0)
            width[m] = 0.4 / slope if slope > 0 else np.nan
    return val, cause, width


def wq(x, w, q):
    # weighted quantile (weights: the harvested area of the crop)
    ok = np.isfinite(x) & (w > 0)
    if not ok.any():
        return np.nan
    o = np.argsort(x[ok]); c = np.cumsum(w[ok][o]); c = c / c[-1]
    return float(np.interp(q, c, x[ok][o]))


print("functions defined")

## 2. Margins per cell, response curves and the runs

For each crop: the scans along ΔT and ΔP (and the 2-D grid), the margins and their causes, the change of each run
(ΔT_run: growing-period temperature, season mean weighted by the phase lengths; ΔP_run: season precipitation, the
same weighting), and the classes crossed / approaching / comfortable. Censored margins (beyond the scanned range) are
set to the edge + one step for the quantiles.

In [ ]:
summary, runs_rows, amoc_rows, curves_T, curves_P, grid_rows, check, CELLS = [], [], [], [], [], [], [], {}
EDGE_T, EDGE_P = DT_GRID[-1] + 0.25, DP_GRID[-1] + 0.05
for crop in CROPS:
    t0 = time.time()
    ctx = context(crop)
    w, I, J = ctx["w"], ctx["I"], ctx["J"]
    RT, RP = scan(ctx, DT_GRID, "dT"), scan(ctx, DP_GRID, "dP")
    k0 = int(np.argmin(np.abs(DT_GRID)))
    check.append({"crop": crop, "presence cells": ctx["n_pres"], "domain cells": len(w),
                  "domain area (Mha)": w.sum() / 1e6, "domain not suitable at 0 (cells)": int((~RT["S"][k0]).sum())})
    m = {}
    m["cold"], m["cause_cold"], m["width_cold"] = margin(DT_GRID, RT, -1, ctx)
    m["warm"], m["cause_warm"], m["width_warm"] = margin(DT_GRID, RT, +1, ctx)
    m["dry"], m["cause_dry"], m["width_dry"] = margin(DP_GRID, RP, -1)
    m["wet"], m["cause_wet"], m["width_wet"] = margin(DP_GRID, RP, +1)
    # the runs in the same units
    e5 = ctx["V"]
    tg_e = season_mean((e5["BIO01"] + e5["BIO02"]) / 2, (e5["BIO06"] + e5["BIO07"]) / 2, ctx["cal"])
    pr_e = season_mean(e5["BIO05"], e5["BIO10"], ctx["cal"])
    dT_run, dP_run = {}, {}
    for run in RUNS:
        with xr.open_dataset(DELTA / f"BIOvariables_delta_{run}_{crop}.nc") as d:
            v = {c: d[c].values[I, J] for c in ("BIO01", "BIO02", "BIO05", "BIO06", "BIO07", "BIO10")}
        dT_run[run] = season_mean((v["BIO01"] + v["BIO02"]) / 2, (v["BIO06"] + v["BIO07"]) / 2, ctx["cal"]) - tg_e
        dP_run[run] = np.where(pr_e > 0.1, season_mean(v["BIO05"], v["BIO10"], ctx["cal"]) / np.maximum(pr_e, 1e-9) - 1, np.nan)
    cold, warm = np.nan_to_num(m["cold"], nan=-np.inf), np.nan_to_num(m["warm"], nan=np.inf)
    dry, wet = np.nan_to_num(m["dry"], nan=-np.inf), np.nan_to_num(m["wet"], nan=np.inf)

    def classes(run):
        t, q = dT_run[run], np.nan_to_num(dP_run[run])
        crossed = (t <= cold) | (t >= warm) | (q <= dry) | (q >= wet)
        near = ~crossed & ((t - cold < NEAR_DT) | (warm - t < NEAR_DT) | (q - dry < NEAR_DP) | (wet - q < NEAR_DP))
        return crossed, near

    for reg, rm in REGIONS.items():
        r_in = rm[I, J]
        wr = w * r_in
        row = {"crop": crop, "region": reg, "today's crop area in the domain (Mha)": wr.sum() / 1e6}
        if wr.sum() > 0:
            for side, edge in (("cold", -EDGE_T), ("warm", EDGE_T), ("dry", -EDGE_P), ("wet", EDGE_P)):
                x = np.where(np.isnan(m[side]), edge, m[side])
                sc = 100 if side in ("dry", "wet") else 1
                for q in (0.25, 0.5, 0.75):
                    row[f"{side} margin q{int(q * 100)}"] = sc * wq(x, wr, q)
                for c, name in CAUSES.items():
                    row[f"{side}: {name} (%)"] = 100 * wr[m[f"cause_{side}"] == c].sum() / wr.sum()
                sharp = SHARP_DT if side in ("cold", "warm") else SHARP_DP
                isp = m[f"cause_{side}"] == 1
                row[f"{side}: median width"] = sc * wq(m[f"width_{side}"], wr * isp, 0.5)
                row[f"{side}: sharp (%)"] = 100 * wr[isp & (m[f"width_{side}"] < sharp)].sum() / wr.sum()
            for run in RUNS:
                crossed, near = classes(run)
                lost6 = ~ctx["S6"][run][I, J] & ~ctx["N6"][run][I, J]
                runs_rows.append({"crop": crop, "region": reg, "run": run,
                                  "dT_run (median, degC)": wq(dT_run[run], wr, 0.5),
                                  "dP_run (median, %)": 100 * wq(dP_run[run], wr, 0.5),
                                  "crossed (%)": 100 * wr[crossed].sum() / wr.sum(),
                                  "approaching (%)": 100 * wr[near].sum() / wr.sum(),
                                  "comfortable (%)": 100 * wr[~crossed & ~near].sum() / wr.sum(),
                                  "lost in 6.b (%)": 100 * wr[lost6].sum() / wr.sum(),
                                  "lost in 6.b and crossed (% of lost)": 100 * wr[lost6 & crossed].sum() / max(wr[lost6].sum(), 1e-9),
                                  "crossed and lost in 6.b (% of crossed)": 100 * wr[lost6 & crossed].sum() / max(wr[crossed].sum(), 1e-9)})
            for h, o in PAIRS:
                ch, _ = classes(h); co, _ = classes(o)
                dTa = dT_run[h] - dT_run[o]
                amoc_rows.append({"crop": crop, "region": reg, "pair": f"{h} - {o}",
                                  "AMOC dT (median, degC)": wq(dTa, wr, 0.5),
                                  "AMOC dP (median, %)": 100 * wq(dP_run[h] - dP_run[o], wr, 0.5),
                                  "crossed only with the hosing (%)": 100 * wr[ch & ~co].sum() / wr.sum(),
                                  "crossed only without it (%)": 100 * wr[co & ~ch].sum() / wr.sum(),
                                  "crossed in both (%)": 100 * wr[ch & co].sum() / wr.sum(),
                                  "AMOC cooling larger than the cold margin left (%)":
                                      100 * wr[~co & (dTa < 0) & (-dTa >= dT_run[o] - cold)].sum() / wr.sum()})
        summary.append(row)
        for k, g in enumerate(DT_GRID):
            p = np.where(RT["N"][k], np.nan, RT["P"][k])
            curves_T.append({"crop": crop, "region": reg, "dT": g, "p q25": wq(p, wr, 0.25), "p median": wq(p, wr, 0.5),
                             "p q75": wq(p, wr, 0.75), "suitable (% of the domain area)": 100 * wr[RT["S"][k]].sum() / max(wr.sum(), 1e-9)})
        for k, g in enumerate(DP_GRID):
            p = np.where(RP["N"][k], np.nan, RP["P"][k])
            curves_P.append({"crop": crop, "region": reg, "dP (%)": 100 * g, "p q25": wq(p, wr, 0.25), "p median": wq(p, wr, 0.5),
                             "p q75": wq(p, wr, 0.75), "suitable (% of the domain area)": 100 * wr[RP["S"][k]].sum() / max(wr.sum(), 1e-9)})
    if DO_2D:
        for a in GRID2_DT:
            for b in GRID2_DP:
                s = climate_at(ctx, a, b)[3]
                for reg, rm in REGIONS.items():
                    wr = w * rm[I, J]
                    if wr.sum() > 0:
                        grid_rows.append({"crop": crop, "region": reg, "dT": a, "dP (%)": 100 * b,
                                          "suitable (% of the domain area)": 100 * wr[s].sum() / wr.sum()})
    # per-cell maps
    dv = {}
    for k_, v_ in m.items():
        g2 = np.full(LAND.shape, np.nan, np.float32); g2[I, J] = v_
        dv[k_] = (("lat", "lon"), g2)
    for run in RUNS:
        for name, arr in (("dT", dT_run[run]), ("dP", dP_run[run])):
            g2 = np.full(LAND.shape, np.nan, np.float32); g2[I, J] = arr
            dv[f"{name}_{run}"] = (("lat", "lon"), g2)
    xr.Dataset(dv, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "set": SET, "causes": str(CAUSES), "dP": "fraction", "level": LEVEL}
               ).to_netcdf(OUT / f"margins_{crop}.nc")
    CELLS[crop] = {"m": m, "I": I, "J": J, "w": w, "dT_run": dT_run}
    print(f"{crop}: {len(w)} cells, {(time.time() - t0) / 60:.1f} min", flush=True)

CHECK = pd.DataFrame(check).set_index("crop")
display(CHECK)
assert (CHECK["domain not suitable at 0 (cells)"] <= 0.01 * CHECK["domain cells"]).all(), \
    "the domain is not suitable at zero shift: the rule of 4.g is not reproduced"
SUM = pd.DataFrame(summary).set_index(["crop", "region"])
RUNS_T = pd.DataFrame(runs_rows).set_index(["crop", "region", "run"])
AMOC_T = pd.DataFrame(amoc_rows).set_index(["crop", "region", "pair"])
CT, CP = pd.DataFrame(curves_T), pd.DataFrame(curves_P)
G2 = pd.DataFrame(grid_rows)
SUM.to_csv(OUT / "margin_summary.csv"); RUNS_T.to_csv(OUT / "runs_vs_margin.csv"); AMOC_T.to_csv(OUT / "amoc_vs_margin.csv")
CT.to_csv(OUT / "curves_dT.csv", index=False); CP.to_csv(OUT / "curves_dP.csv", index=False)
if DO_2D:
    G2.to_csv(OUT / "grid2d.csv", index=False)

## 3. Results

(1) Margins of today's cropland (quartiles weighted by the harvested area; °C for cold / warm, % for dry / wet), their
causes and the sharpness of the threshold. (2) The runs against the margins. (3) The AMOC weakening against the margins.

In [ ]:
with pd.option_context("display.max_rows", 300, "display.max_columns", 60, "display.width", 250):
    cols = [c for c in SUM.columns if c.endswith(("q25", "q50", "q75")) or "median width" in c or "sharp" in c]
    print("(1) margins, global and north of 50 N")
    display(SUM.loc[(slice(None), ["global", "north of 50 N", "tropics"]), ["today's crop area in the domain (Mha)", *cols]].round(2))
    print("    causes of the margin (% of the domain area)")
    display(SUM.loc[(slice(None), "global"), [c for c in SUM.columns if c.endswith("(%)") and "sharp" not in c]].round(1))
    print("(2) the runs against the margins (global)")
    display(RUNS_T.xs("global", level="region").round(1))
    print("(3) the AMOC weakening against the margins")
    display(AMOC_T.xs(PAIRS[-1][0] + " - " + PAIRS[-1][1], level="pair").round(2))

## 4. Figures

(a) Share of today's crop area still suitable against ΔT and ΔP, all crops (global), with the median ΔT_run / ΔP_run
of the 2050 runs. (b) Probability against ΔT per crop: median and quartiles, global and north of 50° N. (c) The 2-D
grid with the four runs. (d) Maps of the cold and warm margins per crop.

In [ ]:
h50, o50 = PAIRS[-1]
colors = plt.cm.tab10(np.linspace(0, 1, 10))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for n, crop in enumerate(CROPS):
    c = CT[(CT.crop == crop) & (CT.region == "global")]
    axes[0].plot(c["dT"], c["suitable (% of the domain area)"], color=colors[n], label=crop.replace("_rf", ""))
    c = CP[(CP.crop == crop) & (CP.region == "global")]
    axes[1].plot(c["dP (%)"], c["suitable (% of the domain area)"], color=colors[n])
    for run, ls in ((o50, "--"), (h50, ":")):
        r = RUNS_T.loc[(crop, "global", run)]
        axes[0].axvline(r["dT_run (median, degC)"], color=colors[n], ls=ls, lw=0.8)
        axes[1].axvline(r["dP_run (median, %)"], color=colors[n], ls=ls, lw=0.8)
axes[0].set_xlabel("uniform shift of the temperature (degC)"); axes[1].set_xlabel("uniform change of the precipitation (%)")
for ax in axes:
    ax.set_ylabel("today's crop area still suitable (%)"); ax.grid(alpha=0.3); ax.axvline(0, color="k", lw=0.8)
axes[0].legend(fontsize=8)
fig.suptitle("Response of today's cropland to uniform shifts of the ERA5 climate (global); dashed: median change of "
             "2050 no hosing, dotted: 2050 hosing", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / "curves_share_suitable.png", dpi=100, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(2, len(CROPS), figsize=(3.2 * len(CROPS), 6.5), squeeze=False, sharey=True)
for n, crop in enumerate(CROPS):
    for r_, reg in enumerate(["global", "north of 50 N"]):
        ax = axes[r_, n]
        c = CT[(CT.crop == crop) & (CT.region == reg)]
        ax.fill_between(c["dT"], c["p q25"], c["p q75"], color="#9ecae1", alpha=0.7)
        ax.plot(c["dT"], c["p median"], color="#08519c")
        ax.axhline(cu.CLIMATE_P, color="r", lw=0.8); ax.axvline(0, color="k", lw=0.8); ax.grid(alpha=0.3)
        ax.set_title(f"{crop.replace('_rf', '')}, {reg}", fontsize=9); ax.set_xlabel("dT (degC)", fontsize=8)
axes[0, 0].set_ylabel("probability"); axes[1, 0].set_ylabel("probability")
fig.suptitle("Probability against a uniform temperature shift: median and quartiles over today's cropland", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / "curves_probability_dT.png", dpi=100, bbox_inches="tight"); plt.show()

if DO_2D:
    fig, axes = plt.subplots(1, len(CROPS), figsize=(3.6 * len(CROPS), 3.6), squeeze=False)
    for n, crop in enumerate(CROPS):
        ax = axes[0, n]
        g = G2[(G2.crop == crop) & (G2.region == "global")].pivot(index="dP (%)", columns="dT", values="suitable (% of the domain area)")
        im = ax.pcolormesh(g.columns, g.index, g.values, vmin=0, vmax=100, cmap="RdYlGn", shading="nearest")
        for run, mk in zip(RUNS, ["o", "s", "o", "s"]):
            r = RUNS_T.loc[(crop, "global", run)]
            ax.plot(r["dT_run (median, degC)"], r["dP_run (median, %)"], mk, mfc="none" if "2025" in run else "k", mec="k")
        ax.set_title(crop.replace("_rf", ""), fontsize=9); ax.set_xlabel("dT (degC)", fontsize=8)
    axes[0, 0].set_ylabel("dP (%)")
    fig.colorbar(im, ax=axes[0, :].tolist(), label="today's crop area still suitable (%)")
    fig.suptitle("Two-dimensional shifts (global); circles: no hosing, squares: hosing; open: 2025, filled: 2050", fontsize=10)
    fig.savefig(OUT / "grid2d.png", dpi=100, bbox_inches="tight"); plt.show()

for crop in CROPS:
    with xr.open_dataset(OUT / f"margins_{crop}.nc") as ds:
        fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
        ds["cold"].plot(ax=axes[0], cmap="Blues_r", vmin=-8, vmax=0, cbar_kwargs={"label": "degC"})
        ds["warm"].plot(ax=axes[1], cmap="Reds", vmin=0, vmax=8, cbar_kwargs={"label": "degC"})
        axes[0].set_title(f"{crop}: cold margin (cooling tolerated; blank where beyond -8 degC or not cultivated)", fontsize=9)
        axes[1].set_title(f"{crop}: warm margin (warming tolerated)", fontsize=9)
        fig.tight_layout(); fig.savefig(OUT / f"{crop}_margin_maps.png", dpi=90); plt.close(fig)
print("written:", sorted(p.name for p in OUT.iterdir()))

## 5. Reading

To be written after the run on BDAP: (1) the margins of each crop and region, and which side and cause dominate; (2)
whether the thresholds are sharp (narrow transition) or gradual; (3) the share of today's cropland that the warming
and the AMOC weakening carry beyond its margin, and how much of the losses of 6.b the uniform margins explain; (4)
the thresholds in °C to quote (e.g. the cooling tolerated by the cropland north of 50° N against the AMOC cooling).